In [1]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from pylate import indexes, models, retrieve
from dotenv import load_dotenv

load_dotenv()

# ==========================================
# 1. LLM (unchanged from before)
# ==========================================
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

# Note: this `embeddings` object is NOT used by ColBERT below -- ColBERT does its
# own token-level embedding internally via the pylate `model`. Kept here only if
# you need single-vector embeddings elsewhere in your pipeline (e.g. RAPTOR).
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

c:\Users\asoha\Desktop\cse\langchain_learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==========================================
# 2. Load the ColBERT model via PyLate
# ==========================================
print("Loading ColBERT model...")
model = models.ColBERT(model_name_or_path="colbert-ir/colbertv2.0")

Loading ColBERT model...


No sentence-transformers model found with name colbert-ir/colbertv2.0.
c:\Users\asoha\Desktop\cse\langchain_learning\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\asoha\.cache\huggingface\hub\models--colbert-ir--colbertv2.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|█

In [3]:
# ==========================================
# 3. Initialize the PLAID index
# ==========================================
index = indexes.PLAID(
    index_folder="pylate-colbert-index",
    index_name="Miyazaki_Studio_Index",
    override=True,  # overwrite if it already exists, e.g. on re-runs
)

retriever_engine = retrieve.ColBERT(index=index)

In [4]:
# ==========================================
# 4. Sample documents + indexing
# ==========================================
my_documents = [
    "In June 1985, Hayao Miyazaki, Isao Takahata, Yasuyoshi Tokuma, and Toshio Suzuki founded the animation production company Studio Ghibli.",
    "Studio Ghibli's first film, Laputa: Castle in the Sky, was released in 1986 and directed by Hayao Miyazaki.",
    "The Disney Renaissance era was highly influenced by competition with the development of Miyazaki's films.",
    "Transformers One is a 2024 American animated science fiction action film directed by Josh Cooley.",
]
documents_ids = [str(i) for i in range(len(my_documents))]

print("Encoding documents...")
documents_embeddings = model.encode(
    my_documents,
    batch_size=32,
    is_query=False,       # critical: documents vs queries are encoded differently
    show_progress_bar=True,
)

print("Indexing documents with ColBERT (PLAID)...")
index.add_documents(
    documents_ids=documents_ids,
    documents_embeddings=documents_embeddings,
)

Encoding documents...


Encoding documents (bs=32): 100%|██████████| 1/1 [00:00<00:00,  4.39it/s]


Indexing documents with ColBERT (PLAID)...


In [5]:
# ==========================================
# 5. Custom retriever function (replaces RAG.as_langchain_retriever)
# ==========================================
# pylate has no built-in LangChain wrapper, so we write the glue ourselves:
# encode the query -> retrieve top-k doc ids/scores -> map ids back to text.
id_to_text = dict(zip(documents_ids, my_documents))

def colbert_retrieve(query: str, k: int = 2) -> str:
    query_embedding = model.encode(
        [query],
        batch_size=1,
        is_query=True,     # critical: must be True for queries
        show_progress_bar=False,
    )
    results = retriever_engine.retrieve(queries_embeddings=query_embedding, k=k)
    # results[0] is a list of {"id": ..., "score": ...} for the first (only) query
    top_docs = [id_to_text[r["id"]] for r in results[0]]
    return "\n\n".join(top_docs)

# Wrap as a LangChain Runnable so it slots into the LCEL chain like a retriever
retriever = RunnableLambda(colbert_retrieve)


In [6]:
# ==========================================
# 6. Prompt (unchanged)
# ==========================================
template = """Answer the question based only on the following context provided by the ColBERT retriever:
Context:
{context}

Question: {question}
Answer:"""

prompt = ChatPromptTemplate.from_template(template)

In [7]:
# ==========================================
# 7. LCEL chain
# ==========================================
colbert_rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# ==========================================
# 8. Run
# ==========================================
if __name__ == "__main__":
    query = "Who were the four founders of Studio Ghibli?"

    print(f"\nRunning query: '{query}'\n")
    response = colbert_rag_chain.invoke(query)

    print("--- LLM Response ---")
    print(response)


Running query: 'Who were the four founders of Studio Ghibli?'

--- LLM Response ---
The four founders of Studio Ghibli were Hayao Miyazaki, Isao Takahata, Yasuyoshi Tokuma, and Toshio Suzuki.
